# Phylogenetic tree

**Objective:**  
Construct a phylogenetic tree using COI sequences and annotate it with life history and taxonomic classes 


**Data used:**
- `LHI_species_COI.csv` : COI sequences with species identifiers
- `pca.csv` : PC1 value for each species
- `species.csv` : Taxonomic information 

## Tree construction

In [2]:
import pandas as pd

# Load data
df_sequences = pd.read_csv('../data/LHI_species_COI.csv')
display(df_sequences.head())

,ID,MaxCompleteness,sequence,species,accession,gene,sample
0,Merluccius_merluccius,1.0,CACTCCTGGGCGACGATCAAATTTATAACGTGATCGTCACGGCACA...,Merluccius merluccius,OL684361,COI-5P,GBMNF46663-22
1,Argyrosomus_japonicus,1.0,AGGTTTATAACGTAATTGTTACGGCGCATGCCTTCGTTATAATTTT...,Argyrosomus japonicus,KJ566657,COI-5P,ANGBF30119-19
2,Mystus_cavasius,1.0,CTATTATTAATATGAAACCCCCAGCCATCTCCCAATATCAAACCCC...,Mystus cavasius,NaN,COI-5P,FBMP029-19
3,Anabas_testudineus,1.0,ACAGCACACGCTTTCGTAATGATTTTCTTTATAGTAATGCCGATGA...,Anabas testudineus,MT511547,COI-5P,GBMNC16885-20
4,Plethodon_cinereus,1.0,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,Plethodon cinereus,EF525825,COI-5P,AMPAS170-05


### Convert CSV to FASTA format

In [3]:
csv_file = "../data/LHI_species_COI.csv"  # Input CSV file
fasta_file = '../tree/sequences_converted.fasta'  # Output FASTA file
id_column = 'ID'  # Column name for sequence IDs
sequence_column = 'sequence'  # Column name for sequences

# Load CSV file
df = pd.read_csv(csv_file)

# Write FASTA file
with open(fasta_file, 'w') as f:
    for index, row in df.iterrows():
        # Write header line (starts with >)
        f.write(f">{row[id_column]}\n")
        # Write sequence line
        f.write(f"{row[sequence_column]}\n")

print(f"Converted {len(df)} sequences to {fasta_file}")

Converted 995 sequences to ../tree/sequences_converted.fasta


### Tree construction   

**Note:**  
The actual tree construction is performed using external tools in Linux environment.

**Files created**:  
- `sequences_converted.fasta` : Raw sequences
- `sequences_aligned.fasta` : FASTA file with aligned sequences   
- `tree_output.newick` : Phylogenetic tree 

## Tree annotation

In [4]:
df_pca = pd.read_csv('../data/pca.csv')
df_species = pd.read_csv('../data/species.csv')

df_species = df_species[['Species', 'Class']]
df_pca = df_pca[['Species', 'PC1']]


# Dataset for annotations 
df_annotation_1 = pd.merge(df_pca, df_species, on='Species', how='inner')
df_annotation_1 = df_annotation_1[['Species', 'PC1', 'Class']] 

df_pc1_annotation = pd.merge(df_sequences, df_pca, left_on='ID', right_on='Species', how='inner')
df_pc1_annotation = df_pc1_annotation[['Species', 'PC1', 'sequence']]
df_annotation = pd.merge(df_pc1_annotation, df_species, on='Species', how='inner')

df_annotation = df_annotation[["Species", "PC1", "Class"]]
display(df_annotation.head())

,Species,PC1,Class
0,Merluccius_merluccius,0.799366,Actinopterygii
1,Argyrosomus_japonicus,0.717958,Actinopterygii
2,Mystus_cavasius,-0.457820,Actinopterygii
3,Anabas_testudineus,-0.907593,Actinopterygii
4,Plethodon_cinereus,-0.536192,Amphibia


In [5]:
# TXT annotation file for taxonomic classes

# Define colors for each taxonomic class (using specified colors)
class_colors = {
    'Ascidiacea': '#e78284',
    'Chondrichthyes': '#8caaee', 
    'Actinopterygii': '#f4b8e4',
    'Amphibia': '#a6d189',
    'Reptilia': '#e5c890',
    'Aves': '#ffa05c',
    'Mammalia': '#808080',
    'Elasmobranchii': "#a3446c",    
    'Gastropoda': '#ffcc00',        
    'Cephalopoda': '#00cccc',      
    'Arachnida': '#804000',         
    'Bivalvia': '#ff8000',          
    'Malacostraca': '#8000ff',      
    'Acoelomorpha': '#888888',      
    'Clitellata': '#888888',        
    'Echinoidea': '#888888',        
    'Eutardigrada': '#888888',      
    'Gastrotricha': '#888888',      
    'Holothuroidea': '#888888',     
    'Leptocardii': '#888888'        
}

# Create TREE_COLORS file
with open('../tree/annotations/annotation_taxonomic_classes.txt', 'w') as f:
    # Header
    f.write("TREE_COLORS\n")
    f.write("SEPARATOR SPACE\n")
    f.write("DATASET_LABEL Taxonomic Classes\n")
    f.write("COLOR #000000\n")
    f.write("\n")
    f.write("DATA\n")

    
    # Create individual label background colors for each species
    for _, row in df_annotation.iterrows():
        species = row['Species']
        tax_class = row['Class']
        color = class_colors.get(tax_class, '#888888')  # Default gray
        
        # Format: NODE_ID range COLOR LABEL
        f.write(f"{species} range {color} {tax_class}\n")

print("TREE_COLORS annotation file created: annotation_taxonomic_classes.txt")

# Summary
class_counts = df_annotation['Class'].value_counts()
print(f"\nSpecies per taxonomic class:")
for class_name, count in class_counts.items():
    print(f"- {class_name}: {count} species")

TREE_COLORS annotation file created: annotation_taxonomic_classes.txt

Species per taxonomic class:
- Actinopterygii: 658 species
- Amphibia: 106 species
- Reptilia: 102 species
- Aves: 36 species
- Chondrichthyes: 29 species
- Malacostraca: 18 species
- Bivalvia: 12 species
- Gastropoda: 8 species
- Clitellata: 5 species
- Eutardigrada: 3 species
- Ascidiacea: 3 species
- Holothuroidea: 2 species
- Mammalia: 2 species
- Arachnida: 2 species
- Leptocardii: 1 species
- Echinoidea: 1 species
- Cephalopoda: 1 species
- Ophiuroidea: 1 species
- Turbellaria: 1 species
- Phoronida: 1 species
- Acoelomorpha: 1 species
- Secernentea: 1 species
- Gastrotricha: 1 species


## Chordata-specific phylogenetic tree

**Objective:**  
Create a focused phylogenetic tree containing only species from the phylum Chordata



In [6]:
# Filter data for Chordata phylum only
chordata_classes = [
    'Actinopterygii', 'Chondrichthyes','Amphibia', 'Reptilia',
    'Aves', 'Mammalia', 'Ascidiacea'
]

df_sequences_with_taxonomy = pd.merge(df_sequences, df_species[['Species', 'Class']], 
                                    left_on='ID', right_on='Species', how='inner')
df_chordata_sequences = df_sequences_with_taxonomy[df_sequences_with_taxonomy['Class'].isin(chordata_classes)]


print(f"Original sequences: {len(df_sequences)} species")
print(f"Chordata sequences: {len(df_chordata_sequences)} species")
print(f"Filtered out: {len(df_sequences_with_taxonomy) - len(df_chordata_sequences)} non-chordate species")

df_chordata_sequences

Original sequences: 995 species
Chordata sequences: 936 species
Filtered out: 59 non-chordate species


,ID,MaxCompleteness,sequence,species,accession,gene,sample,Species,Class
0,Merluccius_merluccius,1.000000,CACTCCTGGGCGACGATCAAATTTATAACGTGATCGTCACGGCACA...,Merluccius merluccius,OL684361,COI-5P,GBMNF46663-22,Merluccius_merluccius,Actinopterygii
1,Argyrosomus_japonicus,1.000000,AGGTTTATAACGTAATTGTTACGGCGCATGCCTTCGTTATAATTTT...,Argyrosomus japonicus,KJ566657,COI-5P,ANGBF30119-19,Argyrosomus_japonicus,Actinopterygii
2,Mystus_cavasius,1.000000,CTATTATTAATATGAAACCCCCAGCCATCTCCCAATATCAAACCCC...,Mystus cavasius,NaN,COI-5P,FBMP029-19,Mystus_cavasius,Actinopterygii
3,Anabas_testudineus,1.000000,ACAGCACACGCTTTCGTAATGATTTTCTTTATAGTAATGCCGATGA...,Anabas testudineus,MT511547,COI-5P,GBMNC16885-20,Anabas_testudineus,Actinopterygii
4,Plethodon_cinereus,1.000000,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,Plethodon cinereus,EF525825,COI-5P,AMPAS170-05,Plethodon_cinereus,Amphibia
...,...,...,...,...,...,...,...,...,...
990,Tauraco_erythrolophus,0.993026,CTATCCATTGGCTTCTTAGGCTTCATTGTATGGGCACATCATATAT...,Tauraco erythrolophus,MN356149_1,COI-5P,NaN,Tauraco_erythrolophus,Aves
991,Speleomantes_ambrosii,1.000000,TATAGTCGGCACAGCTTTAAGTCTCTTAATCCGATCAGAACTTAGC...,Speleomantes ambrosii,12339663,COI-5P,NaN,Speleomantes_ambrosii,Amphibia
992,Thamnophis_saurita,1.000000,ACCCTGTACCTACTATTCGGAGCCTGATCCGGATTAATCGGGGCCT...,Thamnophis saurita,4617324,COI-5P,NaN,Thamnophis_saurita,Reptilia
993,Natrix_maura,1.000000,CCCCCCCTGTCGGGAAATCTAGTACACTCTGGCCCATCAGTAGACT...,Natrix maura,MW478001_1,COI-5P,NaN,Natrix_maura,Reptilia


In [7]:
# Create Chordata FASTA file
fasta_chordata_file = '../tree/sequences_chordata.fasta'

with open(fasta_chordata_file, 'w') as f:
    for index, row in df_chordata_sequences.iterrows():
        # Write header line (starts with >)
        f.write(f">{row['ID']}\n")
        # Write sequence line
        f.write(f"{row['sequence']}\n")

print(f"Converted {len(df_chordata_sequences)} sequences to {fasta_chordata_file}")

Converted 936 sequences to ../tree/sequences_chordata.fasta


### Tree construction   


**Files created**:  
- `sequences_chordata.fasta` : Raw sequences
- `chordata_aligned.fasta` : FASTA file with aligned sequences   
- `tree_chordata.newick` : Phylogenetic tree using FastTree
- `chordata_aligned.phy` : PHYLIP file with aligned sequences 
- `tree_raxml.newick` :  Phylogenetic tree using RAxML



### Tree annotation

In [8]:
# Filter df_annotation for Chordata only
df_chordata_annotation = df_annotation[df_annotation['Class'].isin(chordata_classes)]
df_chordata_annotation

,Species,PC1,Class
0,Merluccius_merluccius,0.799366,Actinopterygii
1,Argyrosomus_japonicus,0.717958,Actinopterygii
2,Mystus_cavasius,-0.457820,Actinopterygii
3,Anabas_testudineus,-0.907593,Actinopterygii
4,Plethodon_cinereus,-0.536192,Amphibia
...,...,...,...
990,Tauraco_erythrolophus,1.922991,Aves
991,Speleomantes_ambrosii,0.917814,Amphibia
992,Thamnophis_saurita,2.577315,Reptilia
993,Natrix_maura,2.009456,Reptilia


In [9]:
# TXT annotation file for taxonomic chordata classes

# Define colors for each taxonomic class (using specified colors)
class_colors = {
    'Ascidiacea': '#e78284',
    'Chondrichthyes': '#8caaee', 
    'Actinopterygii': '#f4b8e4',
    'Amphibia': '#a6d189',
    'Reptilia': '#e5c890',
    'Aves': '#ffa05c',
    'Mammalia': '#808080',      
}

# Create TREE_COLORS file
with open('../tree/annotations/annotation_chordata_classes.txt', 'w') as f:
    # Header
    f.write("TREE_COLORS\n")
    f.write("SEPARATOR SPACE\n")
    f.write("DATASET_LABEL Taxonomic Classes\n")
    f.write("COLOR #000000\n")
    f.write("\n")
    f.write("DATA\n")

    
    # Create individual label background colors for each species
    for _, row in df_chordata_annotation.iterrows():
        species = row['Species']
        tax_class = row['Class']
        color = class_colors.get(tax_class, '#888888')  # Default gray
        
        # Format: NODE_ID range COLOR LABEL
        f.write(f"{species} range {color} {tax_class}\n")


print("TREE_COLORS annotation file created: annotation_chordata_classes.txt")

# Summary
class_counts = df_chordata_annotation['Class'].value_counts()
print(f"\nSpecies per taxonomic class:")
for class_name, count in class_counts.items():
    print(f"- {class_name}: {count} species")



TREE_COLORS annotation file created: annotation_chordata_classes.txt

Species per taxonomic class:
- Actinopterygii: 658 species
- Amphibia: 106 species
- Reptilia: 102 species
- Aves: 36 species
- Chondrichthyes: 29 species
- Ascidiacea: 3 species
- Mammalia: 2 species


In [10]:
# Annotation file for PC1 (life history)

# Create DATASET_GRADIENT file
with open('../tree/annotations/annotation_chordata_pc1.txt', 'w') as f:
    # Header
    f.write("DATASET_GRADIENT\n")
    f.write("SEPARATOR SPACE\n")
    f.write("DATASET_LABEL PC1\n")
    f.write("COLOR #ff0000\n")
    f.write("\n")
    f.write("COLOR_MIN #0000ff\n")
    f.write("COLOR_MAX #ff0000\n")
    f.write("\n")
    f.write("DATA\n")

    
    # Create individual label background colors for each species
    for _, row in df_chordata_annotation.iterrows():
        species = row['Species']
        pc1 = row['PC1']
        
        # Format: NODE_ID range COLOR LABEL
        f.write(f"{species} {pc1}\n")


print("DATASET_GRADIENT annotation file created: annotation_chordata_pc1.txt")


DATASET_GRADIENT annotation file created: annotation_chordata_pc1.txt


## Pagel's Lambda analysis

**Pagel's λ** is a parameter that quantifies the degree of phylogenetic signal in trait evolution:
- **λ = 1**: Traits evolve according to Brownian motion (strong phylogenetic signal)
- **λ = 0**: No phylogenetic signal (traits evolve independently of phylogeny)

### Methods used

This analysis uses three different computational approaches to estimate Pagel's Lambda:

1. **GEIGER::fitContinuous()** - Standard Maximum Likelihood approach (geiger package)
2. **PHYTOOLS::phylosig()** - Alternative implementation with significance testing (phytools package)  
3. **NLME::gls()** - Phylogenetic Generalized Least Squares approach with corPagel() correlation structure (NLME package)

In [ ]:
# Load Pagel's Lambda results
df_lambda_raxml = pd.read_csv('../data/lambda_results_raxml.csv')
df_lambda_fasttree = pd.read_csv('../data/lambda_results_fasttree.csv')


print("=== Pagel's lambda results summary ===")
print(f"Total traits analyzed: {len(df_lambda_raxml)}")
print(f"Traits: {', '.join(df_lambda_raxml['Trait'].tolist())}")
print()

# Display the complete results table
print("=== Lambda comparison tables ===")
display(df_lambda_fasttree)
display(df_lambda_raxml)

=== PAGEL'S LAMBDA RESULTS SUMMARY ===
Total traits analyzed: 14
Traits: PC1, Lb, Li, Lim, Lp, Lpm, Ri, Wwb, Wwi, Wwim, Wwp, ab, am, tp

=== LAMBDA COMPARISON TABLES ===


,Trait,N,GEIGER_Lambda,PHYTOOLS_Lambda,NLME_Lambda
0,PC1,936,0.744087,0.947056,1.015625
1,Lb,214,0.425138,0.999927,1.125000
2,Li,936,0.376417,0.616533,0.589467
3,Lim,180,0.687560,0.687554,1.015625
4,Lp,820,0.462687,0.877134,1.000977
5,Lpm,105,0.621209,0.820960,0.815140
6,Ri,936,0.000000,0.316055,0.500000
7,Wwb,936,0.040546,0.999927,1.001953
8,Wwi,936,0.121834,0.188885,0.500000
9,Wwim,142,0.228833,0.228828,0.233737


,Trait,N,GEIGER_Lambda,PHYTOOLS_Lambda,NLME_Lambda
0,PC1,936,0.953252,0.953247,0.942759
1,Lb,214,0.995739,0.995743,NaN
2,Li,936,0.653286,0.653285,0.615516
3,Lim,180,0.687421,0.687425,0.690485
4,Lp,820,0.860488,0.860503,0.875000
5,Lpm,105,0.807030,0.807043,0.805757
6,Ri,936,0.257288,0.257276,0.252821
7,Wwb,936,0.999998,0.999927,NaN
8,Wwi,936,0.236212,0.236235,0.255448
9,Wwim,142,0.217813,0.217818,0.224873
